# 02 — Momentum Signal Construction

Construct and validate the pre-specified 12–2 momentum signal before any concentration-conditioning test.

**Definition:** at formation month t, compound monthly total returns from t-12 through t-2. Month t-1 is skipped.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles


## 1. Build formation and history panels


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'

clean, audit = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)

print(f'Formation rows: {len(clean):,}')
print(f'History rows: {len(history):,}')
audit


## 2. Construct 12–2 momentum


In [ ]:
mom = attach_momentum_to_formation_universe(clean, history)
mom = assign_signal_quintiles(mom)

coverage = pd.DataFrame({
    'formation_rows': mom.groupby('MthCalDt').size(),
    'momentum_available': mom.groupby('MthCalDt')['Momentum_12_2'].count(),
    'next_return_available': mom.groupby('MthCalDt')['NextMthRet'].count(),
})
coverage['momentum_coverage_pct'] = 100 * coverage['momentum_available'] / coverage['formation_rows']
coverage.tail(12)


## 3. Distribution sanity check


In [ ]:
mom['Momentum_12_2'].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99])


## 4. Inspect winners and losers in selected months


In [ ]:
selected = pd.to_datetime(['2001-12-31','2008-12-31','2020-12-31','2025-12-31'])
available = mom[mom['MthCalDt'].isin(selected)].copy()

cols = [c for c in ['MthCalDt','PERMNO','Ticker','IssuerNm','Momentum_12_2','SignalQuintile','NextMthRet'] if c in available.columns]

top = (available.dropna(subset=['Momentum_12_2'])
       .sort_values(['MthCalDt','Momentum_12_2'], ascending=[True,False])
       .groupby('MthCalDt', group_keys=False).head(10)[cols])
bottom = (available.dropna(subset=['Momentum_12_2'])
          .sort_values(['MthCalDt','Momentum_12_2'], ascending=[True,True])
          .groupby('MthCalDt', group_keys=False).head(10)[cols])

print('TOP MOMENTUM')
display(top)
print('BOTTOM MOMENTUM')
display(bottom)


## 5. Quintile-size check


In [ ]:
quintile_counts = (
    mom.dropna(subset=['SignalQuintile'])
    .groupby(['MthCalDt','SignalQuintile'], observed=True)
    .size()
    .unstack()
)
quintile_counts.tail()


## Stop here before inference

At this stage we only validate signal construction and coverage. Rank IC, portfolio returns, and concentration-conditioning tests come next, after these checks pass.
